# 特征工程

时间特征 · 分箱编码 · 标签编码 · 类别特征定义

In [1]:
import pandas as pd
import numpy as np
import warnings
from logistics_delay.data.loader import load_raw_data_with_target
from logistics_delay.features.engineering import engineer_features, get_feature_lists
from logistics_delay.utils.paths import FIGURES_EDA
warnings.filterwarnings('ignore')
print("[OK] 导入完成")

[OK] 导入完成


In [2]:
# 一键清洗 + 特征工程
df = load_raw_data_with_target()
df = engineer_features(df, run_clean=True, save_processed=True)
print("覆盖旧Excel文件")
FEATURES_ENC, FEATURES_XGB, XGB_CAT_COLS = get_feature_lists()

[loader] 原始数据加载完成: (6880, 32)
[loader] 目标变量构建完成: 延误=4332, 准时=2548
[loader] 延误率: 62.97%
  特征工程管道
  数据清洗管道
[cleaner] 删除 24 行冲突数据 (0.35%)
[cleaner] 清洗冲突后: 6856 行
[cleaner] 原始日期范围: 1899-12-30 ~ 2020-12-03
[cleaner] 过滤掉 2 行非 [2019, 2020] 数据
[cleaner] 保留 6854 行
[cleaner] 运输距离缺失: 705 条
发现 705 条缺失运输距离的记录

填补完成: 705 条记录通过地理邻近性填补
剩余 0 条记录使用中位数填补
[cleaner] 地理填补完成，共填补 705 条
[cleaner] Minimum_kms_to_be_covered_in_a_day: 类别变量，缺失值填充为 UNKNOWN
[cleaner] 基本字段填充完成
[cleaner] ⚠ 仍有缺失值的字段: {'Data_Ping_time': 952, 'Current_Location': 963, 'actual_eta': 37, 'Curr_lat': 952, 'Curr_lon': 952, 'ontime': 4331, 'delay': 2537, 'trip_end_date': 194, 'Driver_Name': 3403, 'Driver_MobileNo': 4163, '_dist_original': 705}
[cleaner] 清洗完成, 形状: (6854, 38), 延误率: 63.19%
[engineering] 特征工程完成, 形状: (6854, 54), 总列数: 54
[engineering] 特征数量: 15 (sklearn) / 15 (XGBoost/CatBoost)
[engineering] 已保存 → C:\Users\a\Desktop\TU-Eindhoven\Logistics_Delay_Project\data\processed\truck_delay_handled_file.xlsx (6854 行 × 54 列)
覆盖旧Excel文件


In [3]:
print("start_weekday:", sorted(df["start_weekday"].unique()))
print("start_month:", sorted(df["start_month"].unique()))
print("planned_days:", df["planned_days"].mean(), "median:", df["planned_days"].median())
print("is_market:", df["is_market"].value_counts().to_dict())
print("supplier_is_large:", df["supplier_is_large"].value_counts().to_dict())

# Minimum_kms_to_be_covered_in_a_day 分布饼图
kms_counts = df["Minimum_kms_to_be_covered_in_a_day"].value_counts().sort_index()
print("\nMinimum_kms_to_be_covered_in_a_day:\n", kms_counts.to_string())

start_weekday: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]
start_month: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
planned_days: 2.195943974321564 median: 2.0
is_market: {0: 6785, 1: 69}
supplier_is_large: {0: 6026, 1: 828}

Minimum_kms_to_be_covered_in_a_day:
 Minimum_kms_to_be_covered_in_a_day
0.0          24
250.0      2528
275.0       264
UNKNOWN    4038


In [4]:
for col in XGB_CAT_COLS:
    print(f"  {col}: {df[col].nunique()} categories")

  vehicleType: 45 categories
  OriginLocation_Code: 179 categories
  DestinationLocation_Code: 477 categories
  GpsProvider: 30 categories
  booking_prefix: 4 categories
  origin_city: 69 categories
  dest_city: 215 categories
  customerID: 39 categories
  Minimum_kms_to_be_covered_in_a_day: 4 categories


In [5]:
print(f"[OK] Final shape: {df.shape}, features: {len(FEATURES_ENC)}")

[OK] Final shape: (6854, 54), features: 15
